In [2]:
import pandas as pd
import numpy as np
import sklearn
import plotly
import seaborn
import yellowbrick

print("All imports successful ✓")
print("pandas:", pd.__version__)
print("sklearn:", sklearn.__version__)

All imports successful ✓
pandas: 3.0.2
sklearn: 1.8.0


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

# ── Paths ──────────────────────────────────────────────
ROOT      = os.path.abspath(os.path.join(os.getcwd(), '..'))
RAW       = os.path.join(ROOT, 'data', 'raw')
PROCESSED = os.path.join(ROOT, 'data', 'processed')

UCI_PATH       = os.path.join(RAW, 'uci',       'online_retail_II.xlsx')
HM_TRANS_PATH  = os.path.join(RAW, 'hm',        'transactions_train.csv')
HM_CUST_PATH   = os.path.join(RAW, 'hm',        'customers.csv')
HM_ART_PATH    = os.path.join(RAW, 'hm',        'articles.csv')
INST_ORD_PATH  = os.path.join(RAW, 'instacart', 'orders.csv')
INST_PROD_PATH = os.path.join(RAW, 'instacart', 'order_products__prior.csv')
INST_NAMES_PATH= os.path.join(RAW, 'instacart', 'products.csv')

print("Paths set ✓")
print("Root:", ROOT)

Paths set ✓
Root: C:\Users\aniru\OneDrive\Desktop\Retail Seg


In [5]:
print("Loading UCI dataset")

uci_1 = pd.read_excel(UCI_PATH, sheet_name='Year 2009-2010', dtype={'Customer ID': str})
uci_2 = pd.read_excel(UCI_PATH, sheet_name='Year 2010-2011', dtype={'Customer ID': str})

uci_raw = pd.concat([uci_1, uci_2], ignore_index=True)

print(f"Shape: {uci_raw.shape}")
print(f"\nColumns: {uci_raw.columns.tolist()}")
print(f"\nDate range: {uci_raw['InvoiceDate'].min()} → {uci_raw['InvoiceDate'].max()}")
print(f"\nNull counts:\n{uci_raw.isnull().sum()}")

Loading UCI dataset
Shape: (1067371, 8)

Columns: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']

Date range: 2009-12-01 07:45:00 → 2011-12-09 12:50:00

Null counts:
Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64


In [6]:
uci_raw.head(3)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom


In [7]:
uci = uci_raw.copy()

# 1. Drop rows with no Customer ID (can't do RFM without a customer)
before = len(uci)
uci = uci.dropna(subset=['Customer ID'])
print(f"Dropped {before - len(uci):,} rows with missing Customer ID")

# 2. Remove cancelled invoices (they start with 'C')
before = len(uci)
uci = uci[~uci['Invoice'].astype(str).str.startswith('C')]
print(f"Dropped {before - len(uci):,} cancelled invoices")

# 3. Remove rows where Quantity or Price is zero or negative
before = len(uci)
uci = uci[(uci['Quantity'] > 0) & (uci['Price'] > 0)]
print(f"Dropped {before - len(uci):,} rows with bad Quantity/Price")

# 4. Remove duplicates
before = len(uci)
uci = uci.drop_duplicates()
print(f"Dropped {before - len(uci):,} duplicate rows")

# 5. Calculate line-level revenue
uci['Revenue'] = uci['Quantity'] * uci['Price']

print(f"\nClean UCI shape: {uci.shape}")
print(f"Unique customers: {uci['Customer ID'].nunique():,}")
print(f"Date range: {uci['InvoiceDate'].min()} → {uci['InvoiceDate'].max()}")

Dropped 243,007 rows with missing Customer ID
Dropped 18,744 cancelled invoices
Dropped 71 rows with bad Quantity/Price
Dropped 26,124 duplicate rows

Clean UCI shape: (779425, 9)
Unique customers: 5,878
Date range: 2009-12-01 07:45:00 → 2011-12-09 12:50:00


In [8]:
print("Loading H&M transactions")

hm_trans = pd.read_csv(HM_TRANS_PATH, dtype={
    'customer_id': str,
    'article_id':  str,
    'price':       np.float32
}, parse_dates=['t_dat'])

hm_cust = pd.read_csv(HM_CUST_PATH, dtype={'customer_id': str})
hm_art  = pd.read_csv(HM_ART_PATH,  dtype={'article_id':  str})

print(f"Transactions shape: {hm_trans.shape}")
print(f"Customers shape:    {hm_cust.shape}")
print(f"Articles shape:     {hm_art.shape}")
print(f"\nTransaction columns: {hm_trans.columns.tolist()}")
print(f"\nDate range: {hm_trans['t_dat'].min()} → {hm_trans['t_dat'].max()}")
print(f"\nNull counts (transactions):\n{hm_trans.isnull().sum()}")

Loading H&M transactions
Transactions shape: (31788324, 5)
Customers shape:    (1371980, 7)
Articles shape:     (105542, 25)

Transaction columns: ['t_dat', 'customer_id', 'article_id', 'price', 'sales_channel_id']

Date range: 2018-09-20 00:00:00 → 2020-09-22 00:00:00

Null counts (transactions):
t_dat               0
customer_id         0
article_id          0
price               0
sales_channel_id    0
dtype: int64


In [9]:
hm_trans.head(3)

,t_dat,customer_id,article_id,price,sales_channel_id
0,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,0663713001,0.050831,2
1,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,0541518023,0.030492,2
2,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,0505221004,0.015237,2


In [10]:
hm = hm_trans.copy()

# 1. Drop nulls
before = len(hm)
hm = hm.dropna(subset=['customer_id', 'price'])
print(f"Dropped {before - len(hm):,} rows with nulls")

# 2. Remove zero/negative prices
before = len(hm)
hm = hm[hm['price'] > 0]
print(f"Dropped {before - len(hm):,} rows with bad price")

# 3. Remove duplicates
before = len(hm)
hm = hm.drop_duplicates()
print(f"Dropped {before - len(hm):,} duplicate rows")

# 4. H&M prices are stored as fractions — multiply by 100 to get real value
# (H&M normalises prices to 0.0x range in this dataset)
hm['price'] = hm['price'] * 100

# 5. H&M has no Quantity column — each row is one item purchased
hm['Quantity'] = 1
hm['Revenue']  = hm['price']

print(f"\nClean H&M shape: {hm.shape}")
print(f"Unique customers: {hm['customer_id'].nunique():,}")
print(f"Date range: {hm['t_dat'].min()} → {hm['t_dat'].max()}")

Dropped 0 rows with nulls
Dropped 0 rows with bad price
Dropped 2,974,905 duplicate rows

Clean H&M shape: (28813419, 7)
Unique customers: 1,362,281
Date range: 2018-09-20 00:00:00 → 2020-09-22 00:00:00


In [11]:
hm.head(3)

,t_dat,customer_id,article_id,price,sales_channel_id,Quantity,Revenue
0,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,0663713001,5.083051,2,1,5.083051
1,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,0541518023,3.049153,2,1,3.049153
2,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,0505221004,1.523729,2,1,1.523729


In [12]:
print("Loading Instacart data...")

inst_orders = pd.read_csv(INST_ORD_PATH, dtype={'user_id': str, 'order_id': str})
inst_prods  = pd.read_csv(INST_PROD_PATH, dtype={'order_id': str, 'product_id': str})
inst_names  = pd.read_csv(INST_NAMES_PATH, dtype={'product_id': str})

print(f"Orders shape:   {inst_orders.shape}")
print(f"Products shape: {inst_prods.shape}")
print(f"Names shape:    {inst_names.shape}")
print(f"\nOrder columns:   {inst_orders.columns.tolist()}")
print(f"Product columns: {inst_prods.columns.tolist()}")

Loading Instacart data...
Orders shape:   (3421083, 7)
Products shape: (32434489, 4)
Names shape:    (49688, 4)

Order columns:   ['order_id', 'user_id', 'eval_set', 'order_number', 'order_dow', 'order_hour_of_day', 'days_since_prior_order']
Product columns: ['order_id', 'product_id', 'add_to_cart_order', 'reordered']


In [13]:
# Keep only 'prior' orders (actual purchase history — not test/train splits)
inst_orders = inst_orders[inst_orders['eval_set'] == 'prior'].copy()

# Join order-level data with product-level data
inst = inst_prods.merge(inst_orders[['order_id','user_id','order_dow',
                                      'order_hour_of_day','days_since_prior_order']],
                         on='order_id', how='left')

# Join product names
inst = inst.merge(inst_names[['product_id','product_name']], on='product_id', how='left')

# Instacart has no price — we assign a proxy unit price of 1.0
# (We'll use order count and basket size for RFM, not real £/$ values)
inst['price']    = 1.0
inst['Quantity'] = 1
inst['Revenue']  = 1.0

# Drop nulls
before = len(inst)
inst = inst.dropna(subset=['user_id'])
print(f"Dropped {before - len(inst):,} rows with missing user_id")

print(f"\nClean Instacart shape: {inst.shape}")
print(f"Unique customers: {inst['user_id'].nunique():,}")

Dropped 0 rows with missing user_id

Clean Instacart shape: (32434489, 12)
Unique customers: 206,209


In [14]:
# ── UCI ───────────────────────────────────────────────
uci_std = uci.rename(columns={
    'Customer ID':  'customer_id',
    'InvoiceDate':  'invoice_date',
    'Invoice':      'order_id',
    'Revenue':      'revenue',
    'Description':  'product_name',
    'Country':      'country'
})[['customer_id', 'invoice_date', 'order_id', 'revenue', 'product_name', 'country']]

uci_std['source'] = 'UCI Online Retail'

# ── H&M ───────────────────────────────────────────────
hm_std = hm.rename(columns={
    'customer_id':  'customer_id',
    't_dat':        'invoice_date',
    'sales_channel_id': 'country',
    'Revenue':      'revenue',
})[['customer_id', 'invoice_date', 'revenue']].copy()

# H&M uses sales channel (1=online, 2=store) instead of country
hm_std['order_id']      = hm['t_dat'].astype(str) + '_' + hm['customer_id']
hm_std['product_name']  = hm['article_id']
hm_std['country']       = 'Sweden'   # H&M is Swedish — used as retail origin label
hm_std['source']        = 'H&M Fashion'

# ── Instacart ─────────────────────────────────────────
inst_std = inst.rename(columns={
    'user_id':      'customer_id',
    'product_name': 'product_name',
    'Revenue':      'revenue',
})[['customer_id', 'revenue', 'product_name']].copy()

# Instacart has no real dates — build a synthetic invoice_date from order sequence
# days_since_prior_order tells us time gaps; we reconstruct approximate dates
inst_std['order_id']     = inst['order_id']
inst_std['country']      = 'USA'
inst_std['source']       = 'Instacart Grocery'

# Reconstruct approximate dates (anchor = 2022-01-01 as synthetic modern baseline)
order_dates = (inst_orders
    .sort_values(['user_id','order_id'])
    .assign(days_cumulative=lambda df:
        df.groupby('user_id')['days_since_prior_order']
          .cumsum().fillna(0))
    .assign(invoice_date=lambda df:
        pd.Timestamp('2022-01-01') + pd.to_timedelta(df['days_cumulative'], unit='D'))
)[['order_id','invoice_date']]

inst_std = inst_std.merge(order_dates, on='order_id', how='left')
inst_std['invoice_date'] = pd.to_datetime(inst_std['invoice_date'])

print("Standardised schemas:")
print(f"  UCI:       {uci_std.shape}  | cols: {uci_std.columns.tolist()}")
print(f"  H&M:       {hm_std.shape}  | cols: {hm_std.columns.tolist()}")
print(f"  Instacart: {inst_std.shape} | cols: {inst_std.columns.tolist()}")

Standardised schemas:
  UCI:       (779425, 7)  | cols: ['customer_id', 'invoice_date', 'order_id', 'revenue', 'product_name', 'country', 'source']
  H&M:       (28813419, 7)  | cols: ['customer_id', 'invoice_date', 'revenue', 'order_id', 'product_name', 'country', 'source']
  Instacart: (32434489, 7) | cols: ['customer_id', 'revenue', 'product_name', 'order_id', 'country', 'source', 'invoice_date']


In [15]:
os.makedirs(PROCESSED, exist_ok=True)

uci_std.to_parquet(os.path.join(PROCESSED, 'uci_clean.parquet'),  index=False)
hm_std.to_parquet(os.path.join(PROCESSED,  'hm_clean.parquet'),   index=False)
inst_std.to_parquet(os.path.join(PROCESSED,'inst_clean.parquet'), index=False)

print("Saved to data/processed/ ✓")
print(f"  uci_clean.parquet  — {len(uci_std):,} rows")
print(f"  hm_clean.parquet   — {len(hm_std):,} rows")
print(f"  inst_clean.parquet — {len(inst_std):,} rows")

Saved to data/processed/ ✓
  uci_clean.parquet  — 779,425 rows
  hm_clean.parquet   — 28,813,419 rows
  inst_clean.parquet — 32,434,489 rows
